In [46]:
# Imports + basic paths for LogLite and Loghub
from pathlib import Path
from collections import deque
import struct


ROOT = Path().resolve()
LOGLITE = ROOT / "loglite" / "scripts"
LOGHUB = ROOT / "dataset" / "loghub"
linux_path = LOGHUB / "Linux" / "Linux_2k.log"

with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    total = sum(1 for _ in f)

# print(f"{ROOT}, {LOGLITE}, {LOGHUB}")

In [47]:
# Parse window.txt into an L-window structure

def load_l_window_from_txt(path: Path):

    """
    Parse a LogLite window dump (text format) into {length: [templates...]}.

    """

    window = {}
    current_len = None
    current_templates = []

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            if not line:
                continue
            if line.startswith("len="):
                # flush previous bucket
                if current_len is not None:
                    window[current_len] = current_templates
                current_len = int(line.split("=", 1)[1])
                current_templates = []
            elif line == "---":
                if current_len is not None:
                    window[current_len] = current_templates
                    current_len = None
                    current_templates = []
            else:
                if current_len is not None:
                    current_templates.append(line)

    if current_len is not None:
        window[current_len] = current_templates
    return window

l_window = load_l_window_from_txt(ROOT / "compressed_logs" / "Linux_2k.window.txt")

# print(l_window)


In [48]:
def keyword_search_loglite_binary_full_decompression(bin_path, query_keywords):
    """
    Keyword search over a LogLite .lite.b file, using the *true*
    bit-level format implemented in LogLite-B (see stream_compress.cc
    and xorc-cli.cc).

    `query_keywords` can be either a single string or an iterable of strings.
    A line is a match only if it contains *all* keywords.

    This mirrors the C++ decompression logic:
    - Read the global boost::dynamic_bitset written by write_bitset_to_file.
    - Walk it entry-by-entry, exactly as xorc-cli's decompression loop does.
    - Reconstruct each original log line using the same sliding window rules.
    - Check each reconstructed line for the query keywords and collect matches.

   
    """

    # Normalize keywords to a list of strings
    if isinstance(query_keywords, str):
        keywords = [query_keywords]
    else:
        keywords = list(query_keywords)

    # Constants from constants.h
    EACH_WINDOW_SIZE_COUNT = 3
    STREAM_ENCODER_COUNT = 13
    ORIGINAL_LENGTH_COUNT = 15
    MAX_LEN = 10000
    RLE_COUNT = 8
    RLE_SKIM = 8
    EACH_WINDOW_SIZE = 1 << EACH_WINDOW_SIZE_COUNT

    results = []

    bin_path = Path(bin_path)
    if not bin_path.exists():
        print(f"Error: File {bin_path} not found.")
        return results

    data = bin_path.read_bytes()
    if len(data) < 16:
        return results

    # Rebuild the boost::dynamic_bitset bitstream written by write_bitset_to_file
    ulong_size = 8  # sizeof(unsigned long) on 64-bit Linux
    size_t_size = 8 # sizeof(size_t)
    file_size = len(data)

    last_block_bits = struct.unpack('<Q', data[file_size - size_t_size : file_size])[0]
    blocks_bytes = data[: file_size - size_t_size]
    if len(blocks_bytes) % ulong_size != 0:
        return results

    num_blocks = len(blocks_bytes) // ulong_size
    blocks = struct.unpack('<' + 'Q' * num_blocks, blocks_bytes)
    bits_per_block = ulong_size * 8
    total_bits = (num_blocks - 1) * bits_per_block + (last_block_bits or bits_per_block)

    pos = 0

    def read_bit():
        nonlocal pos
        if pos >= total_bits:
            return None
        block_idx = pos // bits_per_block
        offset = pos % bits_per_block
        bit = (blocks[block_idx] >> offset) & 1
        pos += 1
        return bit

    def read_int(bit_count: int) -> int:
        v = 0
        for j in range(bit_count):
            b = read_bit()
            if b is None:
                return v
            if b:
                v |= (1 << j)
        return v

    def read_bytes_from_bits(num_bytes: int) -> bytes:
        buf = bytearray(num_bytes)
        for i in range(num_bytes):
            val = read_int(8)
            buf[i] = val & 0xFF
        return bytes(buf)

    # Sliding window used during decompression: length -> deque of recent lines
    window = {}  # type: dict[int, deque[str]]

    while True:
        flag = read_bit()
        if flag is None:
            break

        if flag == 0:
            # Raw log line
            length = read_int(ORIGINAL_LENGTH_COUNT)
            if length <= 0:
                continue
            line_bytes = read_bytes_from_bits(length)
            try:
                line = line_bytes.decode('utf-8', 'ignore')
            except UnicodeDecodeError:
                line = line_bytes.decode('latin1', 'ignore')

            # Update window for this length (reset, as in C++ stream_decompress)
            if length < MAX_LEN:
                dq = deque()
                dq.append(line)
                window[length] = dq

            if all(k in line for k in keywords):
                results.append(line)

        else:
            # Compressed (RLE-encoded XOR against a template in the window)
            window_id = read_int(EACH_WINDOW_SIZE_COUNT)
            len_single_data_bits = read_int(STREAM_ENCODER_COUNT)
            if len_single_data_bits <= 0:
                continue

            # Decode the RLE bitstream into xor_result bytes
            consumed = 0
            xor_bytes = []
            while consumed < len_single_data_bits:
                bit = read_bit()
                if bit is None:
                    break
                consumed += 1

                if bit == 1:
                    # Literal byte: next 8 bits are the byte value
                    val = 0
                    for j in range(8):
                        b = read_bit()
                        if b is None:
                            break
                        consumed += 1
                        if b:
                            val |= (1 << j)
                    xor_bytes.append(val & 0xFF)
                else:
                    # Run of zeros: next RLE_COUNT bits encode the run length
                    zero_count = 0
                    for j in range(RLE_COUNT):
                        b = read_bit()
                        if b is None:
                            break
                        consumed += 1
                        if b:
                            zero_count |= (1 << j)
                    if zero_count > 0:
                        xor_bytes.extend([0] * zero_count)

            if not xor_bytes:
                continue

            xor_result = bytearray(xor_bytes)
            length = len(xor_result)

            dq = window.get(length)
            if not dq or window_id >= len(dq):
                # In a valid stream this should not happen; bail out gracefully.
                continue

            pattern = dq[window_id]
            pattern_bytes = pattern.encode('utf-8', 'ignore')
            if len(pattern_bytes) < length:
                pattern_bytes = pattern_bytes.ljust(length, b'\0')

            # simdReplaceNullCharacters: replace '\0' with pattern character
            for i in range(length):
                if xor_result[i] == 0:
                    xor_result[i] = pattern_bytes[i]

            line_bytes = bytes(xor_result)
            try:
                line = line_bytes.decode('utf-8', 'ignore')
            except UnicodeDecodeError:
                line = line_bytes.decode('latin1', 'ignore')

            # Update window with this newly reconstructed line
            dq = window.setdefault(length, deque())
            if len(dq) < EACH_WINDOW_SIZE:
                dq.append(line)
            else:
                dq.popleft()
                dq.append(line)

            if all(k in line for k in keywords):
                results.append(line)

    return results

In [49]:
def keyword_search_loglite_binary_minor_optimization(bin_path, parsed_l_window, query_keywords):
    """
    Search for keyword(s) in a LogLite compressed binary file (.lite.b)
    using a two-step filtering strategy.

    Step 1 - L-Window Pre-filter:
        Search the parsed L-window templates for entries that contain all
        query keyword(s). Collect the lengths of matching templates into a
        set of 'allowed lengths'. Only log lines whose length matches one
        of these allowed lengths can possibly contain the keyword(s).

    Step 2 - Selective Binary Pass:
        Walk the compressed bitstream entry by entry:

        - Flag 0 (RAW line):
            Read the next 15 bits to get the original string length.
            If this length is in allowed_lengths, read and decompress
            the raw bytes, then perform the keyword check.
            If the length is NOT in allowed_lengths, skip the raw bytes
            entirely (length * 8 bits) without any decoding.

        - Flag 1 (COMPRESSED line):
            Read window_id (3 bits) and rle_bit_len (13 bits).
            Decode the full RLE payload to determine the reconstructed
            string length (sum of all zero-run counts and literal bytes).
            If this length is in allowed_lengths, perform reconstruction
            against the window template and keyword check.
            If NOT in allowed_lengths, skip without reconstruction.
            NOTE: skipped compressed lines do NOT update the window state,
            which may cause incorrect decompression for subsequent lines
            that reference a template built from a skipped line.
            
    Limitation is that decompression would probably be wrong as we are not updating state (l_window) 
    for every log which is required for correctness. The purpose of this is to see how many results we get back 
    with the optimization and to measure performance benefits. Correctness of logs being decompressed correctly would 
    be ensured in later versions.
    """    
    # Constants from constants.h
    EACH_WINDOW_SIZE_COUNT = 3
    STREAM_ENCODER_COUNT = 13
    ORIGINAL_LENGTH_COUNT = 15
    MAX_LEN = 10000
    RLE_COUNT = 8
    EACH_WINDOW_SIZE = 1 << EACH_WINDOW_SIZE_COUNT

    # Normalize keywords
    keywords = [query_keywords] if isinstance(query_keywords, str) else list(query_keywords)
    keywords_bytes = [k.encode('utf-8') for k in keywords]
    results = []

    # --- STEP 2: Search l_window templates for keyword matches ---
    # Store lengths of templates that contain the keyword(s)
    allowed_lengths = set()
    for length, templates in parsed_l_window.items():
        for template in templates:
            if all(k in template for k in keywords):
                allowed_lengths.add(length)
                break  # one match in this length bucket is enough

    # print(f"Allowed lengths from l_window: {sorted(allowed_lengths)}")

    # --- Bitstream Loading ---
    data = Path(bin_path).read_bytes()
    if len(data) < 16:
        return results

    ulong_size = 8
    file_size = len(data)
    last_block_bits = struct.unpack('<Q', data[file_size - 8:])[0]
    blocks_bytes = data[:file_size - 8]
    num_blocks = len(blocks_bytes) // ulong_size
    blocks = struct.unpack('<' + 'Q' * num_blocks, blocks_bytes)
    bits_per_block = 64
    total_bits = (num_blocks - 1) * bits_per_block + (last_block_bits or bits_per_block)

    pos = 0

    def read_bit():
        nonlocal pos
        if pos >= total_bits: return None
        bit = (blocks[pos // 64] >> (pos % 64)) & 1
        pos += 1
        return bit

    def read_int(bit_count):
        v = 0
        for i in range(bit_count):
            b = read_bit()
            if b is not None and b: v |= (1 << i)
        return v

    def skip_bits(n):
        nonlocal pos
        pos += n

    # --- STEP 3: Pass over compressed binary ---
    window = {}  # dict[int, deque[bytes]]

    while True:
        flag = read_bit()
        if flag is None:
            break

        if flag == 0:  # RAW LINE
            length = read_int(ORIGINAL_LENGTH_COUNT)

            if length in allowed_lengths:
                # Decompress and check
                line_bytes = bytearray(length)
                for i in range(length):
                    line_bytes[i] = read_int(8)
                line_bytes = bytes(line_bytes)

                if length < MAX_LEN:
                    window[length] = deque([line_bytes], maxlen=EACH_WINDOW_SIZE)

                if all(k in line_bytes for k in keywords_bytes):
                    results.append(line_bytes.decode('utf-8', 'ignore'))
            else:
                # Skip raw bytes entirely
                skip_bits(length * 8)

        else:  # COMPRESSED LINE
            window_id = read_int(EACH_WINDOW_SIZE_COUNT)
            rle_bit_len = read_int(STREAM_ENCODER_COUNT)

            # Decode RLE payload to find original length
            xor_delta = bytearray()
            bits_consumed = 0
            while bits_consumed < rle_bit_len:
                rle_type = read_bit()
                bits_consumed += 1
                if rle_type == 1:  # Literal
                    xor_delta.append(read_int(8))
                    bits_consumed += 8
                else:  # Zero Run
                    count = read_int(RLE_COUNT)
                    bits_consumed += RLE_COUNT
                    xor_delta.extend(b'\x00' * count)

            length = len(xor_delta)

            if length in allowed_lengths:
                # Decompress and check
                dq = window.get(length)
                if not dq or window_id >= len(dq):
                    continue

                template_bytes = dq[window_id]
                reconstructed = bytearray(xor_delta)
                for i in range(length):
                    if reconstructed[i] == 0:
                        reconstructed[i] = template_bytes[i]
                final_bytes = bytes(reconstructed)

                dq.append(final_bytes)

                if all(k in final_bytes for k in keywords_bytes):
                    results.append(final_bytes.decode('utf-8', 'ignore'))
            else:
                # Skip — but window state is NOT updated
                # This will cause wrong decompression for later lines
                # of this length if they reference a skipped template
                pass

    return results

In [ ]:
# Query 1: simple keyword search for word kernel (uncompressed vs compressed)

# Uncompressed baseline
with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    matches = [line.rstrip("\n") for line in f if "kernel" in line]

q1_matches_uncompressed = len(matches)
print("Query 1 - keyword 'kernel' (uncompressed):", q1_matches_uncompressed, "matches")

# Compressed-domain search over LogLite binary
bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary_minor_optimization(bin_path, l_window, "kernel")
q1_matches_compressed = len(compressed_matches)
print("Query 1 - keyword 'kernel' (compressed):", q1_matches_compressed, "matches")

# Query 1: confusion matrix and metrics

actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")


Query 1 - keyword 'kernel' (uncompressed): 77 matches
Query 1 - keyword 'kernel' (compressed): 77 matches
TP: 77
FP: 0
FN: 0
TN: 1923
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 score:  1.0000


In [51]:
# Query 2: variable value match for PID 28842


with linux_path.open("r", encoding="utf-8", errors="ignore")  as f:
    matches = [line.rstrip("\n") for line in f if "28842" in line]

q2_matches_uncompressed = len(matches)
print("Query 2 - PID 28842 (Uncompressed):", q2_matches_uncompressed, "matches")


bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary_minor_optimization(bin_path, l_window, "28842")
q2_matches_compressed = len(compressed_matches)
print("Query 2 - PID 28842 (Compressed):", q2_matches_compressed, "matches")


# confusion matrix and metrics

actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")


Query 2 - PID 28842 (Uncompressed): 1 matches
Query 2 - PID 28842 (Compressed): 0 matches
TP: 0
FP: 0
FN: 1
TN: 1999
Accuracy:  0.9995
Precision: 0.0000
Recall:    0.0000
F1 score:  0.0000


In [52]:
# Query 3: Time stamp based filtering.

with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    matches = [line.rstrip("\n") for line in f if line.startswith("Jul  2")]

q3_matches_uncompressed = len(matches)
print("Query 3 - Timestamp 'Jul 2' (Uncompressed):", q3_matches_uncompressed, "matches")



bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary_minor_optimization(bin_path, l_window, "Jul  2")
q3_matches_compressed = len(compressed_matches)
print("Query 3 - Timestamp 'Jul 2' (Compressed):", q3_matches_compressed, "matches")


actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")



Query 3 - Timestamp 'Jul 2' (Uncompressed): 41 matches
Query 3 - Timestamp 'Jul 2' (Compressed): 10 matches
TP: 10
FP: 0
FN: 31
TN: 1959
Accuracy:  0.9845
Precision: 1.0000
Recall:    0.2439
F1 score:  0.3922


In [53]:
# Query 4: substring search for failed

with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    matches = [line.rstrip("\n") for line in f if "failed" in line]
    
q4_matches_uncompressed = len(matches)
print("Query 4 - keyword 'failed' (Uncompressed):", q4_matches_uncompressed, "matches")


bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary_minor_optimization(bin_path, l_window, "failed")
q4_matches_compressed = len(compressed_matches)
print("Query 4 - keyword 'failed' (Compressed):", q4_matches_compressed, "matches")



actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")



Query 4 - keyword 'failed' (Uncompressed): 47 matches
Query 4 - keyword 'failed' (Compressed): 24 matches
TP: 24
FP: 0
FN: 23
TN: 1953
Accuracy:  0.9885
Precision: 1.0000
Recall:    0.5106
F1 score:  0.6761


In [54]:
# Query 5: multi-token serach of tokens not next to each other

with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    matches = []
    for raw in f:
        line = raw.rstrip("\n")
        if "sshd" in line and "failure" in line:
            matches.append(line)
            
q5_matches_uncompressed = len(matches)
print("Query 5 - 'sshd' and 'failure' (Uncompressed):", q5_matches_uncompressed, "matches")


bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary_minor_optimization(bin_path, l_window, ["sshd","failure"])
q5_matches_compressed = len(compressed_matches)
print("Query 5 - 'sshd' and 'failure' (Compressed):", q5_matches_compressed, "matches")


actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")



Query 5 - 'sshd' and 'failure' (Uncompressed): 489 matches
Query 5 - 'sshd' and 'failure' (Compressed): 475 matches
TP: 475
FP: 0
FN: 14
TN: 1511
Accuracy:  0.9930
Precision: 1.0000
Recall:    0.9714
F1 score:  0.9855
